In [2]:
import pandas as pd

df = pd.read_csv("/home/sayantan/Documents/credit_loan_risk/data/application_train.csv")
print("Shape:", df.shape)
print(df['TARGET'].value_counts(normalize=True))

Shape: (307511, 122)
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [3]:
# the answer key + a few columns we care about
cols = ['TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']
df[cols].describe()

,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,DAYS_BIRTH,DAYS_EMPLOYED
count,307511.000000,3.075110e+05,3.075110e+05,307499.000000,307511.000000,307511.000000
mean,0.080729,1.687979e+05,5.990260e+05,27108.573909,-16036.995067,63815.045904
std,0.272419,2.371231e+05,4.024908e+05,14493.737315,4363.988632,141275.766519
min,0.000000,2.565000e+04,4.500000e+04,1615.500000,-25229.000000,-17912.000000
25%,0.000000,1.125000e+05,2.700000e+05,16524.000000,-19682.000000,-2760.000000
50%,0.000000,1.471500e+05,5.135310e+05,24903.000000,-15750.000000,-1213.000000
75%,0.000000,2.025000e+05,8.086500e+05,34596.000000,-12413.000000,-289.000000
max,1.000000,1.170000e+08,4.050000e+06,258025.500000,-7489.000000,365243.000000


In [4]:
# build the hero feature: EMI-to-income ratio
df['emi_to_income'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# compare: do defaulters carry heavier EMI burden than repayers?
df.groupby('TARGET')['emi_to_income'].median()

/tmp/ipykernel_1084/357575364.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['emi_to_income'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']


TARGET
0    0.162280
1    0.169294
Name: emi_to_income, dtype: float64

In [5]:
# age in years (fix the negative-days weirdness)
df['age_years'] = -df['DAYS_BIRTH'] / 365

# do younger people default more?
df.groupby('TARGET')['age_years'].median()

/tmp/ipykernel_1084/1006898660.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_years'] = -df['DAYS_BIRTH'] / 365


TARGET
0    43.498630
1    39.128767
Name: age_years, dtype: float64

In [6]:
# how strongly does each number column move WITH default?
corr = df.corr(numeric_only=True)['TARGET'].sort_values()

# the 5 that most PREDICT repayment, and 5 that most predict default
print("Most protective (negative = less default):")
print(corr.head(5))
print("\nMost risky (positive = more default):")
print(corr.tail(6))

Most protective (negative = less default):
EXT_SOURCE_3    -0.178919
EXT_SOURCE_2    -0.160472
EXT_SOURCE_1    -0.155317
age_years       -0.078239
DAYS_EMPLOYED   -0.044932
Name: TARGET, dtype: float64

Most risky (positive = more default):
DAYS_ID_PUBLISH                0.051457
DAYS_LAST_PHONE_CHANGE         0.055218
REGION_RATING_CLIENT           0.058899
REGION_RATING_CLIENT_W_CITY    0.060893
DAYS_BIRTH                     0.078239
TARGET                         1.000000
Name: TARGET, dtype: float64


In [7]:
import numpy as np

# work on a fresh copy so we don't wreck the original
data = df.copy()

# MONSTER 1: the 365243 "1000-year employee" junk code → make it NaN (honest missing)
data['DAYS_EMPLOYED'] = data['DAYS_EMPLOYED'].replace(365243, np.nan)

# MONSTER 2: the income monster (₹11.7cr) → cap extreme incomes at the 99th percentile
income_cap = data['AMT_INCOME_TOTAL'].quantile(0.99)
data['AMT_INCOME_TOTAL'] = data['AMT_INCOME_TOTAL'].clip(upper=income_cap)

print("Employment junk gone? Max now:", data['DAYS_EMPLOYED'].max())
print("Income cap set at: ₹", round(income_cap))
print("New max income: ₹", round(data['AMT_INCOME_TOTAL'].max()))

Employment junk gone? Max now: 0.0
Income cap set at: ₹ 472500
New max income: ₹ 472500


In [8]:
print("Min:", data['DAYS_EMPLOYED'].min(), "| Max:", data['DAYS_EMPLOYED'].max(), "| Missing now:", data['DAYS_EMPLOYED'].isna().sum())

Min: -17912.0 | Max: 0.0 | Missing now: 55374


In [9]:
# === BEHAVIORAL FEATURE FORGE ===

# 1. EMI-to-income — your hero. How stretched is this borrower? (now cleaner post-cap)
data['emi_to_income'] = data['AMT_ANNUITY'] / data['AMT_INCOME_TOTAL']

# 2. Credit-to-income — over-leverage. Loan size vs what they earn.
data['credit_to_income'] = data['AMT_CREDIT'] / data['AMT_INCOME_TOTAL']

# 3. Age in years — your loud signal from Phase 1
data['age_years'] = -data['DAYS_BIRTH'] / 365

# 4. Employment stability — fraction of life spent in current job
data['employment_ratio'] = data['DAYS_EMPLOYED'] / data['DAYS_BIRTH']

# 5. Income per family member — real disposable strain
data['income_per_person'] = data['AMT_INCOME_TOTAL'] / data['CNT_FAM_MEMBERS']

# sanity check: do these split defaulters from repayers?
features = ['emi_to_income', 'credit_to_income', 'employment_ratio', 'income_per_person']
data.groupby('TARGET')[features].median()

,emi_to_income,credit_to_income,employment_ratio,income_per_person
TARGET,,,,
0,0.162375,3.279414,0.121592,75000.0
1,0.169350,3.257143,0.093461,70500.0


In [10]:
def engineer_features(df):
    """Takes raw Home Credit rows, returns clean behavioral features. Used in BOTH training and the API."""
    d = df.copy()

    # --- clean the monsters ---
    d['DAYS_EMPLOYED'] = d['DAYS_EMPLOYED'].replace(365243, np.nan)

    # --- behavioral features ---
    d['emi_to_income']     = d['AMT_ANNUITY'] / d['AMT_INCOME_TOTAL']
    d['credit_to_income']  = d['AMT_CREDIT'] / d['AMT_INCOME_TOTAL']
    d['age_years']         = -d['DAYS_BIRTH'] / 365
    d['employment_ratio']  = d['DAYS_EMPLOYED'] / d['DAYS_BIRTH']
    d['income_per_person'] = d['AMT_INCOME_TOTAL'] / d['CNT_FAM_MEMBERS']

    return d

# test it on the raw data
test = engineer_features(df)
print("New columns exist?", [c for c in ['emi_to_income','age_years','employment_ratio'] if c in test.columns])
print("Shape:", test.shape)

New columns exist? ['emi_to_income', 'age_years', 'employment_ratio']
Shape: (307511, 127)
